# Chunking in LangChain

Once documents are loaded (see `l22_document_loaders.ipynb`), they're usually too big to embed or feed to a model as a single unit: a 296-page PDF loaded as one giant string won't fit in an embedding model's context window, and even if it did, a single fuzzy vector for the whole document is useless for retrieving a specific fact. **Chunking** (splitting) breaks documents into smaller pieces — small enough to embed and retrieve individually, but large enough to keep their content coherent.

Every splitter below lives in `langchain_text_splitters` and shares the same core interface:

- `split_text(text)` — split a raw string into a list of strings
- `split_documents(docs)` / `create_documents(texts)` — split `Document` objects while carrying `metadata` forward
- `chunk_size` — target size per chunk (unit depends on the splitter: characters, tokens, ...)
- `chunk_overlap` — how much adjacent chunks repeat, so context isn't lost at a cut point

This notebook covers the strategies LangChain documents at [docs.langchain.com/oss/python/integrations/splitters](https://docs.langchain.com/oss/python/integrations/splitters): text-structure-based, length-based, and document-structure-based (Markdown, code, JSON).

In [2]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

sample_text = open("data/notes.md").read()
print(sample_text)

c:\Users\hp\Desktop\genai python\genai_class\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
[transformers] PyTorch was not found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.


# RAG Pipeline Notes

Quick reference notes on building a retrieval-augmented generation pipeline.

## Stages

1. **Load** â€” read raw files (PDF, CSV, TXT, MD) into `Document` objects.
2. **Split** â€” chunk long documents so they fit an embedding model's context window.
3. **Embed** â€” turn each chunk into a vector.
4. **Retrieve** â€” find the chunks most similar to a query.
5. **Generate** â€” pass retrieved chunks to an LLM as context.

## Things to watch for

- Chunk size and overlap change retrieval quality a lot.
- Always keep `source` metadata so answers can be traced back to a file.
- CSV rows and PDF pages need different chunking strategies than free text.

> A document loader's only job is step 1 â€” everything downstream assumes it produced clean, well-tagged `Document` objects.



## 1. The default choice — `RecursiveCharacterTextSplitter`

For most use cases, start here. It tries a list of separators in order (`\n\n`, `\n`, `" "`, `""` by default) — splitting on paragraphs first, then falling back to sentences, then words, then characters — so it keeps larger semantic units (paragraphs) intact whenever they fit inside `chunk_size`, and only breaks them apart when it has to.

In [3]:
splitter = RecursiveCharacterTextSplitter(chunk_size=200, chunk_overlap=20)
chunks = splitter.split_text(sample_text)

print(f"{len(sample_text)} characters -> {len(chunks)} chunks\n")
for i, chunk in enumerate(chunks):
    print(f"--- chunk {i} ({len(chunk)} chars) ---")
    print(chunk)
    print()

805 characters -> 7 chunks

--- chunk 0 (109 chars) ---
# RAG Pipeline Notes

Quick reference notes on building a retrieval-augmented generation pipeline.

## Stages

--- chunk 1 (162 chars) ---
1. **Load** â€” read raw files (PDF, CSV, TXT, MD) into `Document` objects.
2. **Split** â€” chunk long documents so they fit an embedding model's context window.

--- chunk 2 (172 chars) ---
3. **Embed** â€” turn each chunk into a vector.
4. **Retrieve** â€” find the chunks most similar to a query.
5. **Generate** â€” pass retrieved chunks to an LLM as context.

--- chunk 3 (22 chars) ---
## Things to watch for

--- chunk 4 (129 chars) ---
- Chunk size and overlap change retrieval quality a lot.
- Always keep `source` metadata so answers can be traced back to a file.

--- chunk 5 (75 chars) ---
- CSV rows and PDF pages need different chunking strategies than free text.

--- chunk 6 (125 chars) ---
> A document loader's only job is step 1 â€” everything downstream assumes it produced clean, wel

Notice the overlap: the tail of one chunk reappears at the start of the next (`chunk_overlap=20`). This protects against a fact getting cut exactly at a chunk boundary and never fully appearing in any single retrieved chunk.

### `split_documents` + `add_start_index`

In a real pipeline you split `Document` objects (from a loader), not raw strings, so `metadata` (like `source`) carries over to every chunk. Setting `add_start_index=True` additionally records each chunk's character offset in the original document — useful for citing exactly where a chunk came from.

In [4]:
from langchain_community.document_loaders import PyMuPDFLoader

docs = PyMuPDFLoader("data/ai_summit.pdf").load()
print(f"Loaded {len(docs)} page(s), {sum(len(d.page_content) for d in docs)} characters total")

splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50, add_start_index=True)
chunks = splitter.split_documents(docs)

print(f"Split into {len(chunks)} chunks")
print(chunks[0].page_content[:200])
print(chunks[0].metadata)

C:\Users\hp\AppData\Local\Temp\ipykernel_9584\3585468051.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyMuPDFLoader


Loaded 7 page(s), 20635 characters total
Split into 50 chunks
India-AI Impact Summit 2026 
 
Welfare for All, Happiness of All 
20 January, 2026 
Key Takeaways 
• 
The India–AI Impact Summit 2026 will be the first global AI summit to be hosted in the Global Sout
{'producer': '', 'creator': 'Microsoft Word', 'creationdate': '2026-01-20T13:24:54+00:00', 'source': 'data/ai_summit.pdf', 'file_path': 'data/ai_summit.pdf', 'total_pages': 7, 'format': 'PDF 1.7', 'title': '', 'author': 'Rishika Basani', 'subject': '', 'keywords': '', 'moddate': '2026-01-20T13:24:54+00:00', 'trapped': '', 'modDate': "D:20260120132454+00'00'", 'creationDate': "D:20260120132454+00'00'", 'page': 0, 'start_index': 0}


## 2. Chunk size is a tradeoff

Smaller chunks are more precise (a retrieved chunk stays on-topic) but lose surrounding context; larger chunks keep context but dilute the embedding with less-relevant text and retrieve less precisely. There's no universally "correct" size — it depends on the embedding model and how questions will be asked. The cell below makes the tradeoff concrete by splitting the same PDF at three sizes.

In [5]:
for size in (200, 500, 1500):
    test_splitter = RecursiveCharacterTextSplitter(chunk_size=size, chunk_overlap=int(size * 0.1))
    test_chunks = test_splitter.split_documents(docs)
    avg_len = sum(len(c.page_content) for c in test_chunks) / len(test_chunks)
    print(f"chunk_size={size:5d} -> {len(test_chunks):3d} chunks, avg {avg_len:.0f} chars/chunk")

chunk_size=  200 -> 157 chunks, avg 131 chars/chunk
chunk_size=  500 ->  50 chunks, avg 422 chars/chunk
chunk_size= 1500 ->  18 chunks, avg 1224 chars/chunk


In [6]:
print(f"  Sample chunk (first one):\n  {test_chunks[0].page_content}\n")
print("-" * 50)

  Sample chunk (first one):
  India-AI Impact Summit 2026 
 
Welfare for All, Happiness of All 
20 January, 2026 
Key Takeaways 
• 
The India–AI Impact Summit 2026 will be the first global AI summit to be hosted in the Global South. 
• 
It will be held from 16 to 20 February 2026 at Bharat Mandapam, New Delhi, as a five-day programme covering 
policy, research, industry, and public engagement. 
• 
It will be anchored on 3 foundational pillars, or 'Sutras': People, Planet and Progress. 
• 
The India AI Impact Expo is expected to feature over 400 exhibitors across more than seven thematic pavilions, 
with participation of over 1.5 lakh visitors. 
Introduction 
Artificial Intelligence (AI) is a key enabler of India’s development journey, 
strengthening governance and improving public service delivery. For India, AI 
supports inclusive growth at scale and aligns with the vision of Viksit Bharat by 
2047. India’s linguistic and cultural diversity positions it strongly for advancing 
multi-l

## 3. `CharacterTextSplitter` — single-separator splitting

`CharacterTextSplitter` is `RecursiveCharacterTextSplitter`'s simpler ancestor: it splits on **one** separator (default `\n\n`) and does *not* fall back to other separators if a piece is still too big — it just leaves that piece oversized. Use it when you know your text has one reliable natural boundary (e.g. blank-line-separated log entries) and want strict, predictable splits rather than the recursive fallback behavior.

In [7]:
from langchain_text_splitters import CharacterTextSplitter

char_splitter = CharacterTextSplitter(separator="\n\n", chunk_size=200, chunk_overlap=0)
char_chunks = char_splitter.split_text(sample_text)

print(f"{len(char_chunks)} chunks (some may exceed chunk_size, since there's no fallback separator)")
for c in char_chunks:
    print(f"  {len(c)} chars")

Created a chunk of size 335, which is longer than the specified 200
Created a chunk of size 205, which is longer than the specified 200


5 chunks (some may exceed chunk_size, since there's no fallback separator)
  109 chars
  335 chars
  22 chars
  205 chars
  125 chars


## 4. Token-based splitting

`chunk_size` in characters doesn't map cleanly to what a model actually sees — LLMs and embedding models bill and truncate by **tokens**, not characters. Both `CharacterTextSplitter` and `RecursiveCharacterTextSplitter` expose `.from_tiktoken_encoder(...)`, which measures `chunk_size` in tokens (via OpenAI's `tiktoken` library) while still cutting text at character boundaries.

In [ ]:
token_splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
    encoding_name="cl100k_base",
    chunk_size=50,
    chunk_overlap=0,
)
token_chunks = token_splitter.split_text(sample_text)

import tiktoken
enc = tiktoken.get_encoding("cl100k_base")

print(f"{len(token_chunks)} chunks, target 50 tokens each\n")
for c in token_chunks:
    print(f"  {len(enc.encode(c)):3d} tokens | {len(c):3d} chars | {c[:60]!r}...")t

6 chunks, target 50 tokens each

   22 tokens | 109 chars | '# RAG Pipeline Notes\n\nQuick reference notes on building a re'...
   47 tokens | 162 chars | '1. **Load** â€” read raw files (PDF, CSV, TXT, MD) into `Doc'...
   32 tokens | 108 chars | '3. **Embed** â€” turn each chunk into a vector.\n4. **Retriev'...
   18 tokens |  63 chars | '5. **Generate** â€” pass retrieved chunks to an LLM as conte'...
   49 tokens | 229 chars | '## Things to watch for\n\n- Chunk size and overlap change retr'...
   29 tokens | 125 chars | "> A document loader's only job is step 1 â€” everything down"...


## 5. Splitting source code — `Language`-aware separators

Generic separators (`\n\n`, `\n`, `" "`) split code in the middle of functions or classes. `RecursiveCharacterTextSplitter.from_language(Language.PYTHON, ...)` swaps in separators tuned to Python's grammar (`\nclass `, `\ndef `, `\n\tdef `, ...), so it prefers to cut between functions/classes rather than inside them. Support exists for dozens of languages — see `Language` for the full list (JS, Java, Go, Markdown, HTML, LaTeX, ...).

In [9]:
from langchain_text_splitters import Language

print([lang.value for lang in Language])

['cpp', 'go', 'java', 'kotlin', 'js', 'ts', 'php', 'proto', 'python', 'r', 'rst', 'ruby', 'rust', 'scala', 'swift', 'markdown', 'latex', 'html', 'sol', 'csharp', 'cobol', 'c', 'lua', 'perl', 'haskell', 'elixir', 'powershell', 'visualbasic6']


In [10]:
python_code = '''
def load_data(path):
    """Load a CSV file into a list of dicts."""
    import csv
    with open(path) as f:
        return list(csv.DictReader(f))


class Aggregator:
    """Running mean/count over numeric values."""

    def __init__(self):
        self.total = 0
        self.count = 0

    def add(self, value):
        self.total += value
        self.count += 1

    def mean(self):
        return self.total / self.count if self.count else 0
'''

python_splitter = RecursiveCharacterTextSplitter.from_language(
    language=Language.PYTHON, chunk_size=120, chunk_overlap=0
)
code_chunks = python_splitter.split_text(python_code)

print(f"{len(code_chunks)} chunks\n")
for i, c in enumerate(code_chunks):
    print(f"--- chunk {i} ---\n{c}")

6 chunks

--- chunk 0 ---
def load_data(path):
    """Load a CSV file into a list of dicts."""
    import csv
    with open(path) as f:
--- chunk 1 ---
return list(csv.DictReader(f))
--- chunk 2 ---
class Aggregator:
    """Running mean/count over numeric values."""
--- chunk 3 ---
def __init__(self):
        self.total = 0
        self.count = 0
--- chunk 4 ---
def add(self, value):
        self.total += value
        self.count += 1
--- chunk 5 ---
def mean(self):
        return self.total / self.count if self.count else 0


## 6. Markdown — splitting by headers

Markdown has explicit structure (`#`, `##`, `###`) that's more meaningful than a raw character count. `MarkdownHeaderTextSplitter` splits on header boundaries and attaches each section's headers as `metadata`, so a chunk retrieved later still knows which section it came from — even after the header text itself is stripped out of `page_content`.

In [11]:
from langchain_text_splitters import MarkdownHeaderTextSplitter

headers_to_split_on = [
    ("#", "h1"),
    ("##", "h2"),
]

md_splitter = MarkdownHeaderTextSplitter(headers_to_split_on=headers_to_split_on)
md_chunks = md_splitter.split_text(sample_text)

print(f"{len(md_chunks)} sections\n")
for c in md_chunks:
    print(f"metadata={c.metadata}")
    print(c.page_content[:150])
    print()

3 sections

metadata={'h1': 'RAG Pipeline Notes'}
Quick reference notes on building a retrieval-augmented generation pipeline.

metadata={'h1': 'RAG Pipeline Notes', 'h2': 'Stages'}
1. **Load** â€” read raw files (PDF, CSV, TXT, MD) into `Document` objects.
2. **Split** â€” chunk long documents so they fit an embedding model's con

metadata={'h1': 'RAG Pipeline Notes', 'h2': 'Things to watch for'}
- Chunk size and overlap change retrieval quality a lot.
- Always keep `source` metadata so answers can be traced back to a file.
- CSV rows and PDF p



Because each `Document` from `MarkdownHeaderTextSplitter` can still be long, it's common to chain it with `RecursiveCharacterTextSplitter` — split by structure first, then by size within each section.

In [12]:
final_splitter = RecursiveCharacterTextSplitter(chunk_size=150, chunk_overlap=20)
final_chunks = final_splitter.split_documents(md_chunks)

print(f"{len(md_chunks)} sections -> {len(final_chunks)} size-bounded chunks\n")
for c in final_chunks:
    print(f"{c.metadata} | {len(c.page_content)} chars")

3 sections -> 7 size-bounded chunks

{'h1': 'RAG Pipeline Notes'} | 76 chars
{'h1': 'RAG Pipeline Notes', 'h2': 'Stages'} | 75 chars
{'h1': 'RAG Pipeline Notes', 'h2': 'Stages'} | 134 chars
{'h1': 'RAG Pipeline Notes', 'h2': 'Stages'} | 124 chars
{'h1': 'RAG Pipeline Notes', 'h2': 'Things to watch for'} | 129 chars
{'h1': 'RAG Pipeline Notes', 'h2': 'Things to watch for'} | 75 chars
{'h1': 'RAG Pipeline Notes', 'h2': 'Things to watch for'} | 125 chars


## 7. JSON — `RecursiveJsonSplitter`

JSON shouldn't be chunked by character count either — cutting a string in half would produce invalid, unparseable fragments. `RecursiveJsonSplitter` walks the JSON structure itself (dicts and lists) and only splits at object/array boundaries, so every chunk stays valid JSON.

In [21]:
from langchain_text_splitters import RecursiveJsonSplitter

sample_json = {
    "summit": "India-AI Impact Summit 2026",
    "location": "Bharat Mandapam, New Delhi",
    "sessions": [
        {"title": "Welfare for All", "speakers": ["A. Sharma", "R. Basani"], "duration_min": 45},
        {"title": "AI Safety Frameworks", "speakers": ["K. Rao"], "duration_min": 60},
        {"title": "Global South AI Access", "speakers": ["M. Iyer", "T. Singh", "P. Nair"], "duration_min": 50},
    ],
}

json_splitter = RecursiveJsonSplitter(max_chunk_size=20)
json_chunks = json_splitter.split_text(sample_json)

print(f"{len(json_chunks)} chunks\n")


for c in json_chunks: 
     print(f"len={len(str(c))}: {c}")

   

2 chunks

len=83: {"summit": "India-AI Impact Summit 2026", "location": "Bharat Mandapam, New Delhi"}
len=288: {"sessions": [{"title": "Welfare for All", "speakers": ["A. Sharma", "R. Basani"], "duration_min": 45}, {"title": "AI Safety Frameworks", "speakers": ["K. Rao"], "duration_min": 60}, {"title": "Global South AI Access", "speakers": ["M. Iyer", "T. Singh", "P. Nair"], "duration_min": 50}]}


## Summary

| Splitter | Splits on | Chunk size unit | Use when |
|---|---|---|---|
| `RecursiveCharacterTextSplitter` | paragraphs → sentences → words → chars (recursive fallback) | characters (or tokens via `.from_tiktoken_encoder`) | Default choice for generic text |
| `CharacterTextSplitter` | one fixed separator, no fallback | characters (or tokens) | Text with one reliable, known boundary |
| `RecursiveCharacterTextSplitter.from_language` | language-aware boundaries (`def`, `class`, ...) | characters | Source code |
| `MarkdownHeaderTextSplitter` | `#`/`##`/`###` headers | N/A (returns sections, not size-bounded) | Markdown — usually chained into a size-based splitter after |
| `RecursiveJsonSplitter` | dict/list boundaries | approx. characters, valid-JSON guaranteed | JSON payloads |

**Rule of thumb:** if the data has explicit structure (Markdown, JSON, code, HTML), split on that structure first, then run a size-based splitter over the resulting sections. For everything else, `RecursiveCharacterTextSplitter` with `chunk_overlap` around 10–20% of `chunk_size` is a solid starting point — the next step is embedding these chunks (`l20_embeddings_cosine_similarity.ipynb`) and storing them for retrieval (`l21_rag_pipeline_from_scratch.ipynb`).